# Notebook 11 — Extractor de recomendaciones con IA (NER local)

**Proyecto BME513 · Universidad de Valparaíso** — Sebastián Inostroza Hurtado

---
## Objetivo

Entrenar un **modelo de extracción** (no de predicción) que **localice la recomendación clínica dentro del informe**, como alternativa/complemento al extractor por reglas (regex).

La regex funciona bien en formatos conocidos, pero exige una regla nueva por cada variación léxica de informes externos. Un modelo aprende el **contexto** de lo que es una recomendación y generaliza a redacciones no vistas.

### Lo que haremos
1. **Auto-etiquetado sin trabajo manual**: la columna `Recommendations` indica la recomendación; la localizamos dentro del `Full_Report` y generamos etiquetas por palabra (esquema BIO).
2. **Entrenar un NER** con **DistilBETO** (Transformer en español), localmente.
3. **Dos pruebas**: (A) *solo extraer* con el modelo + clasificar con el clasificador de reglas existente; (B) *extraer y clasificar*, ambos con modelos.

### Principios que cuidamos (para la defensa)
- **Sin data leakage**: separamos train/val/test **antes** de cualquier ajuste; el test se toca **una sola vez**; solo el `Full_Report` es entrada.
- **Hiperparámetros sólidos**: learning rate, warmup, weight decay y *early stopping* estándar; `max_length` suficiente para no truncar la recomendación (va al final del informe).
- **Todo local**: DistilBETO corre en tu Mac (MPS). Ningún dato sale de la máquina (Ley 19.628).

## 1. Preparación del entorno

In [1]:
# Ejecuta una sola vez si te faltan paquetes:
# !pip install -q transformers datasets seqeval accelerate torch
import transformers, torch
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("MPS disponible:", torch.backends.mps.is_available())

/Users/sebas/Documents/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


transformers: 4.57.6
torch: 2.8.0
MPS disponible: True


### Semillas y reproducibilidad
Fijamos todas las semillas: parte de una evaluación honesta y reproducible.

In [2]:
import random, numpy as np, torch, os
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.backends.mps.is_available(): torch.mps.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print("Dispositivo:", DEVICE)

Dispositivo: mps


## 2. Cargar el dataset
El CSV ya está en el proyecto. Desde `notebooks/`, la ruta es `../data/processed/reports_cleaned.csv`.

Solo usamos dos columnas:
- `Full_Report_clean` → **entrada** del modelo.
- `Recommendations_clean` → **solo para derivar etiquetas**. Nunca es entrada.

In [3]:
import pandas as pd
CSV_PATH = "../data/processed/reports_cleaned.csv"
df = pd.read_csv(CSV_PATH).dropna(subset=["Recommendations_clean"]).reset_index(drop=True)
print("Informes con recomendación:", len(df))
df[["Full_Report_clean", "Recommendations_clean", "BI-RADS"]].head(3)

Informes con recomendación: 4347


,Full_Report_clean,Recommendations_clean,BI-RADS
0,mamografia digital bilateral craneo-caudal y m...,- se sugiere ecografia mamaria y correlacion c...,0
1,mamografia digital bilateral craneo-caudal y m...,- se sugiere correlacion con ecografia mamaria...,2
2,mamografia digital bilateral craneo-caudal y m...,- se sugiere correlacion con ecografia mamaria...,0


## 3. Auto-etiquetado: de la columna a etiquetas BIO

### ¿Qué es el esquema BIO?
Etiquetamos cada palabra con:
- **B-REC** (*Begin*): primera palabra de la recomendación.
- **I-REC** (*Inside*): palabras siguientes de la recomendación.
- **O** (*Outside*): no es recomendación.

Ejemplo: *"mamografia normal . se sugiere control anual ."* → `O O O B-REC I-REC I-REC I-REC O`

### Auto-generación
Como la recomendación aparece **textualmente y una sola vez** en el informe, localizamos esa subcadena y marcamos sus palabras. Cero trabajo manual.

In [4]:
def limpiar_rec(rec):
    return str(rec).strip().lstrip("-*\u2022 ").strip()

def etiquetar_bio(full_report, recomendacion):
    # Devuelve (tokens, etiquetas_bio) a nivel de PALABRA.
    full = str(full_report); rec = limpiar_rec(recomendacion)
    tokens = full.split(); etiquetas = ["O"] * len(tokens)
    rec_tokens = rec.split()
    if not rec_tokens: return tokens, etiquetas
    n, m = len(tokens), len(rec_tokens)
    for i in range(n - m + 1):
        if all(tokens[i+j].strip(".,;:") == rec_tokens[j].strip(".,;:") for j in range(m)):
            etiquetas[i] = "B-REC"
            for j in range(1, m): etiquetas[i+j] = "I-REC"
            break
    return tokens, etiquetas

ejemplos = []; sin_alinear = 0
for _, row in df.iterrows():
    toks, labs = etiquetar_bio(row["Full_Report_clean"], row["Recommendations_clean"])
    if "B-REC" not in labs: sin_alinear += 1; continue
    ejemplos.append({"tokens": toks, "ner_tags": labs, "birads": int(row["BI-RADS"])})
print(f"Ejemplos etiquetados: {len(ejemplos)} | No alineados: {sin_alinear}")
for t, l in list(zip(ejemplos[0]["tokens"], ejemplos[0]["ner_tags"]))[-12:]:
    print(f"  {l:7s} {t}")

Ejemplos etiquetados: 4345 | No alineados: 2
  B-REC   se
  I-REC   sugiere
  I-REC   ecografia
  I-REC   mamaria
  I-REC   y
  I-REC   correlacion
  I-REC   con
  I-REC   estudios
  I-REC   anteriores
  I-REC   para
  I-REC   posterior
  I-REC   recategorizacion.


### Verificación del auto-etiquetado
Comprobamos que el span reconstruido desde las etiquetas coincide con la recomendación original.

In [5]:
import re
def _norm_cmp(s):
    # normaliza para comparar: minúsculas, sin puntuación, espacios colapsados
    return re.sub(r"\s+", " ", re.sub(r"[^a-zñáéíóú0-9 ]", " ", str(s).lower())).strip()

def reconstruir_span(tokens, tags):
    return " ".join(t for t, l in zip(tokens, tags) if l != "O")

ok = 0
for ej, (_, row) in zip(ejemplos, df.iterrows()):
    span = _norm_cmp(reconstruir_span(ej["tokens"], ej["ner_tags"]))
    rec  = _norm_cmp(limpiar_rec(row["Recommendations_clean"]))
    if rec and (rec in span or span in rec):
        ok += 1
print(f"Spans reconstruidos OK: {ok}/{len(ejemplos)} ({100*ok/len(ejemplos):.1f}%)")
print("(comparación robusta: ignora puntuación y espacios)")

Spans reconstruidos OK: 1941/4345 (44.7%)
(comparación robusta: ignora puntuación y espacios)


## 4. Separación train / validación / test (ANTES de tokenizar)

**Este es el paso más importante para evitar data leakage.** Separamos los datos **antes** de cualquier ajuste o tokenización. El conjunto de **test se reserva y no se toca hasta la evaluación final**.

- **70% train** / **15% validación** / **15% test**.
- **Estratificado por BI-RADS**: cada partición mantiene la proporción de categorías, para no dejar sin representación a las clases raras (BI-RADS 4, 5).

### 4.0 Deduplicación robusta (crítico contra data leakage)

**Auditoría de leakage:** el corpus tiene informes duplicados **exactos (~11%)** y también **casi-duplicados** (informes que difieren solo en un número o una palabra; ~20% de similitud alta en muestreo). Si variantes del mismo informe caen en *train* y *test*, el modelo las reconoce y la métrica se **infla**.

Además, el corpus es muy **homogéneo**: la recomendación está al final del informe en el 99% de los casos, y existen solo ~250 recomendaciones distintas (una aparece 783 veces). Por eso es esperable un F1 muy alto **dentro** del corpus — pero eso mide la facilidad del corpus, no la capacidad de generalizar.

Para una métrica interna más honesta, deduplicamos por una **firma** del informe (texto sin números ni signos), eliminando exactos y casi-duplicados antes de separar.

In [6]:
# Deduplicación robusta ANTES del split: exactos Y casi-duplicados.
# Un corpus con informes casi idénticos repartidos entre train/test infla la
# métrica. Usamos una "firma" del informe (texto normalizado, sin dígitos ni
# espacios extra) para agrupar variantes casi iguales y quedarnos con una sola.
import re
def firma(tokens):
    t = " ".join(tokens).lower()
    t = re.sub(r"\d+", "#", t)          # neutraliza números (fechas, medidas)
    t = re.sub(r"[^a-zñáéíóú# ]", " ", t) # solo letras
    t = re.sub(r"\s+", " ", t).strip()
    return t

vistos = set(); ejemplos_unicos = []
for e in ejemplos:
    f = firma(e["tokens"])
    if f in vistos:
        continue
    vistos.add(f)
    ejemplos_unicos.append(e)

print(f"Antes de deduplicar:  {len(ejemplos)}")
print(f"Después (exactos + casi-duplicados): {len(ejemplos_unicos)}")
print(f"Eliminados: {len(ejemplos) - len(ejemplos_unicos)}")
ejemplos = ejemplos_unicos

Antes de deduplicar:  4345
Después (exactos + casi-duplicados): 3765
Eliminados: 580


In [7]:
from sklearn.model_selection import train_test_split
birads_all = [e["birads"] for e in ejemplos]
# 70 / 30, luego 30 -> 15/15
train, temp = train_test_split(ejemplos, test_size=0.30, random_state=SEED,
                               stratify=birads_all)
val, test = train_test_split(temp, test_size=0.50, random_state=SEED,
                             stratify=[e["birads"] for e in temp])
print(f"train={len(train)}  val={len(val)}  test={len(test)}")
# Verificar estratificación
import collections
for nombre, part in [("train",train),("val",val),("test",test)]:
    d = collections.Counter(e["birads"] for e in part)
    print(f"  {nombre}: BI-RADS { {k:d[k] for k in sorted(d)} }")

train=2635  val=565  test=565
  train: BI-RADS {0: 650, 1: 290, 2: 1587, 3: 61, 4: 36, 5: 11}
  val: BI-RADS {0: 139, 1: 62, 2: 341, 3: 13, 4: 7, 5: 3}
  test: BI-RADS {0: 140, 1: 62, 2: 340, 3: 13, 4: 8, 5: 2}


## 5. Tokenización y alineación de etiquetas a subtokens

DistilBETO parte las palabras en **subtokens** (p. ej. "mamográfico" → "mamo","##gráf","##ico"). Debemos alinear las etiquetas BIO de palabra a subtoken con dos reglas estándar:
- Solo el **primer subtoken** de cada palabra lleva la etiqueta real; los subtokens siguientes reciben `-100` (que la función de pérdida ignora), para no contar la misma palabra varias veces.
- Los tokens especiales ([CLS], [SEP], padding) también reciben `-100`.

`max_length=384`: cubre el informe más largo (244 palabras ≈ 320 subtokens) **sin truncar el final**, donde está la recomendación.

In [8]:
from transformers import AutoTokenizer
MODELO = "dccuchile/distilbert-base-spanish-uncased"  # DistilBETO
MAX_LEN = 384
tokenizer = AutoTokenizer.from_pretrained(MODELO)

label_list = ["O", "B-REC", "I-REC"]
label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

def tokenizar_y_alinear(ejemplo):
    tok = tokenizer(ejemplo["tokens"], truncation=True, max_length=MAX_LEN,
                    is_split_into_words=True)
    word_ids = tok.word_ids()
    labels = []; prev = None
    for wid in word_ids:
        if wid is None:
            labels.append(-100)                       # token especial
        elif wid != prev:
            labels.append(label2id[ejemplo["ner_tags"][wid]])  # primer subtoken
        else:
            labels.append(-100)                       # subtoken de continuación
        prev = wid
    tok["labels"] = labels
    return tok

from datasets import Dataset
ds_train = Dataset.from_list(train).map(tokenizar_y_alinear)
ds_val   = Dataset.from_list(val).map(tokenizar_y_alinear)
ds_test  = Dataset.from_list(test).map(tokenizar_y_alinear)
print("Tokenización lista. Ejemplo de longitudes:",
      len(ds_train[0]["input_ids"]), "subtokens")

Map:   0%|          | 0/2635 [00:00<?, ? examples/s]

Map:   0%|          | 0/565 [00:00<?, ? examples/s]

Map:   0%|          | 0/565 [00:00<?, ? examples/s]

Tokenización lista. Ejemplo de longitudes: 159 subtokens


## 6. Modelo y métricas

Cargamos DistilBETO para **clasificación de tokens** (3 etiquetas). La métrica correcta para NER es **a nivel de span** (no de token): `seqeval` calcula precisión, recall y F1 sobre entidades completas, que es lo que nos importa (¿extrajo la recomendación entera bien?).

In [9]:
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification
import numpy as np, evaluate

modelo = AutoModelForTokenClassification.from_pretrained(
    MODELO, num_labels=len(label_list), id2label=id2label, label2id=label2id)
collator = DataCollatorForTokenClassification(tokenizer)
metrica = evaluate.load("seqeval")

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    true_preds, true_labels = [], []
    for pred, lab in zip(preds, labels):
        tp = [id2label[pr] for pr, l in zip(pred, lab) if l != -100]
        tl = [id2label[l]  for pr, l in zip(pred, lab) if l != -100]
        true_preds.append(tp); true_labels.append(tl)
    r = metrica.compute(predictions=true_preds, references=true_labels)
    return {"precision": r["overall_precision"], "recall": r["overall_recall"],
            "f1": r["overall_f1"], "accuracy": r["overall_accuracy"]}

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at dccuchile/distilbert-base-spanish-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 7. Entrenamiento con hiperparámetros cuidados y *early stopping*

Elecciones y su porqué (defendibles):
- **learning_rate = 3e-5**: rango estándar para fine-tuning de BERT (2e-5–5e-5). Más alto desestabiliza; más bajo aprende lento.
- **epochs = 6 con early stopping (patience=2)**: entrenamos hasta 6 épocas pero **paramos si la F1 de validación deja de mejorar**, cargando el mejor checkpoint. Esto **evita el sobreajuste**.
- **weight_decay = 0.01**: regularización L2, reduce overfitting.
- **warmup_ratio = 0.1**: calienta el learning rate al inicio, estabiliza.
- **batch_size = 16**: equilibrio memoria/estabilidad en Apple Silicon.
- **metric_for_best_model = "f1"** y **load_best_model_at_end=True**: seleccionamos el modelo por su F1 de validación, no por la última época.

Nota: el **test NO participa** en ninguna de estas decisiones. Solo `val` guía el early stopping.

In [10]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

args = TrainingArguments(
    output_dir="../models/ner_recomendacion",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    report_to="none",
    use_mps_device=(DEVICE == "mps"),
)

trainer = Trainer(
    model=modelo, args=args,
    train_dataset=ds_train, eval_dataset=ds_val,
    tokenizer=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()

/Users/sebas/Documents/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/transformers/training_args.py:2301: UserWarning: `use_mps_device` is deprecated and will be removed in version 5.0 of 🤗 Transformers. `mps` device will be used by default if available similar to the way `cuda` device is used.Therefore, no action from user is required. 
  warnings.warn(
/var/folders/3r/zv202tcx70l9blnhmnpbw1kc0000gn/T/ipykernel_97184/4246734306.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.
/Users/sebas/Documents/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarn

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.002100,0.001619,0.982578,0.998230,0.990342,0.999681
2,0.001400,0.000414,0.998230,0.998230,0.998230,0.999984
3,0.000900,0.000201,1.000000,1.000000,1.000000,1.000000
4,0.000400,0.000170,1.000000,1.000000,1.000000,1.000000
5,0.000300,0.000152,1.000000,1.000000,1.000000,1.000000


/Users/sebas/Documents/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/sebas/Documents/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/sebas/Documents/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/sebas/Documents/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is s

TrainOutput(global_step=825, training_loss=0.033994100291394824, metrics={'train_runtime': 376.7285, 'train_samples_per_second': 41.967, 'train_steps_per_second': 2.628, 'total_flos': 965330020890900.0, 'train_loss': 0.033994100291394824, 'epoch': 5.0})

## 8. Evaluación final en el conjunto de TEST

Ahora —y solo ahora— tocamos el test, que el modelo nunca vio y que no influyó en ninguna decisión. Esta es la cifra honesta de desempeño del extractor.

In [11]:
resultados_test = trainer.evaluate(ds_test)
print("=== Desempeño en TEST (nunca visto) ===")
for k in ["eval_precision","eval_recall","eval_f1","eval_accuracy"]:
    print(f"  {k.replace('eval_',''):10s}: {resultados_test[k]:.4f}")

/Users/sebas/Documents/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


=== Desempeño en TEST (nunca visto) ===
  precision : 0.9982
  recall    : 1.0000
  f1        : 0.9991
  accuracy  : 1.0000


## 9. Función de inferencia: extraer la recomendación de un informe nuevo

Envuelve el modelo en una función simple: recibe el texto del informe y devuelve el span de recomendación extraído.

In [12]:
import torch
def extraer_recomendacion_ner(texto_informe):
    palabras = str(texto_informe).split()
    tok = tokenizer(palabras, truncation=True, max_length=MAX_LEN,
                    is_split_into_words=True, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        logits = modelo(**tok).logits
    preds = logits.argmax(-1)[0].tolist()
    word_ids = tok.word_ids()
    etiquetas_palabra = {}
    for idx, wid in enumerate(word_ids):
        if wid is not None and wid not in etiquetas_palabra:
            etiquetas_palabra[wid] = id2label[preds[idx]]
    span = [palabras[w] for w in sorted(etiquetas_palabra)
            if etiquetas_palabra[w] != "O"]
    return " ".join(span)

# Prueba con un informe con redacción NO vista ("amerita", "examen")
ejemplo_nuevo = ("mamografia bilateral. hallazgos benignos. birads 2. "
                 "dado el patron mamografico, amerita complementar examen con ecografia.")
print("Recomendación extraída por el NER:")
print(" ", extraer_recomendacion_ner(ejemplo_nuevo))

Recomendación extraída por el NER:
  amerita complementar examen con ecografia.


## 10. PRUEBA A — Extraer con el modelo + clasificar con el clasificador de reglas existente

Esta es la variante **híbrida**: el NER encuentra la recomendación (robusto a redacciones nuevas) y el **clasificador de reglas que ya tenemos** le asigna la categoría (transparente y auditable). Lo mejor de ambos mundos.

In [13]:
import sys, re
sys.path.insert(0, "..")  # para importar src desde notebooks/
from src.extractor_recomendacion import clasificar_recomendacion, _normalizar_texto

# Gatillos que marcan el INICIO real de la recomendación. Si el NER incluyó
# preámbulo ("dado el patron mamografico, se sugiere..."), recortamos desde el
# gatillo para que el clasificador de reglas reciba el texto limpio.
_GATILLOS = r"(se sugiere|se recomienda|recomiendo|recomendamos|sugerimos|amerita|" \
            r"debe|deben|realizar|control|biopsia|derivar|derivacion|complementar|correlacion)"

def _limpiar_span(span):
    m = re.search(_GATILLOS, span, flags=re.IGNORECASE)
    return span[m.start():] if m else span

def pipeline_A(texto_informe):
    span = extraer_recomendacion_ner(texto_informe)       # IA extrae
    span_limpio = _limpiar_span(span)                     # recorta preámbulo
    norm, _ = _normalizar_texto(span_limpio)
    clas = clasificar_recomendacion(norm, es_ya_normalizado=True)  # reglas clasifican
    return {"span_extraido": span, "span_clasificado": span_limpio,
            "categoria": clas["categoria_principal"], "metodo": clas["metodo"]}

print("PRUEBA A (NER extrae + reglas clasifican):")
for caso in informes_chilenos if "informes_chilenos" in dir() else []:
    pass
print(pipeline_A(ejemplo_nuevo))

PRUEBA A (NER extrae + reglas clasifican):
{'span_extraido': 'amerita complementar examen con ecografia.', 'span_clasificado': 'amerita complementar examen con ecografia.', 'categoria': 'ambigua', 'metodo': None}


### Evaluación de la Prueba A sobre el test
Comparamos la categoría obtenida por el pipeline híbrido contra la categoría de referencia (la que el clasificador de reglas asigna al span verdadero).

In [14]:
from src.extractor_recomendacion import clasificar_recomendacion, _normalizar_texto

# Evaluación: categoría del span PREDICHO por el NER vs categoría del span REAL.
# Mide si el pipeline híbrido (NER + reglas) asigna la misma categoría que se
# obtendría con la extracción perfecta.
def categoria_de(texto):
    norm, _ = _normalizar_texto(texto)
    return clasificar_recomendacion(norm, es_ya_normalizado=True)["categoria_principal"]

aciertos = 0
for e in test:
    span_pred = extraer_recomendacion_ner(" ".join(e["tokens"]))
    span_real = reconstruir_span(e["tokens"], e["ner_tags"])
    if categoria_de(span_pred) == categoria_de(span_real):
        aciertos += 1
print(f"Concordancia de categoria (Prueba A) en test: "
      f"{aciertos}/{len(test)} ({100*aciertos/len(test):.1f}%)")

Concordancia de categoria (Prueba A) en test: 565/565 (100.0%)


## 11. PRUEBA B — Extraer y clasificar, ambos con modelos

Aquí el modelo también asigna la **categoría**. Como no tenemos etiquetas de categoría hechas a mano, generamos etiquetas *silver* (plateadas) con el clasificador de reglas sobre el span real, y entrenamos un segundo modelo (clasificación de secuencias) para replicarlas.

**Honestidad metodológica**: estas etiquetas provienen de las reglas, así que el modelo B, en el mejor caso, aprende a imitar al clasificador de reglas. Es útil como demostración de un extractor+clasificador end-to-end, pero **no supera** a las reglas: aprende de ellas.

In [15]:
# Generar etiquetas silver de categoría con el clasificador de reglas
from src.extractor_recomendacion import clasificar_recomendacion, _normalizar_texto
CATS = ["biopsia_histologia","derivacion_oncologica","estudio_complementario_imagen",
        "correlacion_ecografica","comparacion_estudios_previos","control_corto_plazo",
        "control_anual","criterio_medico","ambigua"]
cat2id = {c:i for i,c in enumerate(CATS)}

def silver_cat(tokens, tags):
    span = reconstruir_span(tokens, tags)
    norm,_ = _normalizar_texto(span)
    return clasificar_recomendacion(norm, es_ya_normalizado=True)["categoria_principal"] or "ambigua"

def a_dataset_clasif(part):
    filas = []
    for e in part:
        filas.append({"text": " ".join(e["tokens"]),
                      "label": cat2id[silver_cat(e["tokens"], e["ner_tags"])]})
    return filas

clf_train = a_dataset_clasif(train)
clf_val   = a_dataset_clasif(val)
clf_test  = a_dataset_clasif(test)
import collections
print("Distribución de categorías (train, silver):",
      dict(collections.Counter(f["label"] for f in clf_train)))

Distribución de categorías (train, silver): {6: 777, 2: 554, 3: 1139, 7: 84, 5: 35, 0: 40, 4: 6}


In [16]:
from transformers import AutoModelForSequenceClassification
from datasets import Dataset

tok_clf = AutoTokenizer.from_pretrained(MODELO)
def tok_text(b): return tok_clf(b["text"], truncation=True, max_length=MAX_LEN)
dtr = Dataset.from_list(clf_train).map(tok_text, batched=True)
dva = Dataset.from_list(clf_val).map(tok_text, batched=True)
dte = Dataset.from_list(clf_test).map(tok_text, batched=True)

modelo_clf = AutoModelForSequenceClassification.from_pretrained(MODELO, num_labels=len(CATS))

import numpy as np, evaluate
f1m = evaluate.load("f1")
def metrics_clf(p):
    preds = np.argmax(p.predictions, axis=1)
    return f1m.compute(predictions=preds, references=p.label_ids, average="macro")

from transformers import DataCollatorWithPadding
args_clf = TrainingArguments(
    output_dir="../models/clasificador_recomendacion",
    learning_rate=3e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
    num_train_epochs=6, weight_decay=0.01, warmup_ratio=0.1,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1", greater_is_better=True, seed=SEED,
    report_to="none", use_mps_device=(DEVICE=="mps"))
trainer_clf = Trainer(model=modelo_clf, args=args_clf, train_dataset=dtr, eval_dataset=dva,
    tokenizer=tok_clf, data_collator=DataCollatorWithPadding(tok_clf),
    compute_metrics=metrics_clf, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer_clf.train()
print("Test (Macro F1) clasificador silver:", trainer_clf.evaluate(dte))

Map:   0%|          | 0/2635 [00:00<?, ? examples/s]

Map:   0%|          | 0/565 [00:00<?, ? examples/s]

Map:   0%|          | 0/565 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at dccuchile/distilbert-base-spanish-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/sebas/Documents/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/transformers/training_args.py:2301: UserWarning: `use_mps_device` is deprecated and will be removed in version 5.0 of 🤗 Transformers. `mps` device will be used by default if available similar to the way `cuda` device is used.Therefore, no action from user is required. 
  warnings.warn(
/var/folders/3r/zv202tcx70l9blnhmnpbw1kc0000gn/T/ipykernel_97184/731665501.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_clf = Trainer(model=mo

Epoch,Training Loss,Validation Loss,F1
1,No log,0.094963,0.743502
2,No log,0.056073,0.939094
3,No log,0.036623,0.934691
4,0.333300,0.040372,0.934691


/Users/sebas/Documents/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/sebas/Documents/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/sebas/Documents/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/sebas/Documents/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is s

Test (Macro F1) clasificador silver: {'eval_loss': 0.09230555593967438, 'eval_f1': 0.826297613463673, 'eval_runtime': 3.4272, 'eval_samples_per_second': 164.859, 'eval_steps_per_second': 10.504, 'epoch': 4.0}


## 12. Comparación y conclusiones

Compara honestamente las dos pruebas:

| Enfoque | Extracción | Clasificación | Ventaja | Cuidado |
|---|---|---|---|---|
| **Prueba A** (recomendada) | NER (IA) | reglas existentes | robusta y auditable | mantiene la tabla de reglas |
| **Prueba B** | NER (IA) | modelo (IA) | end-to-end | aprende de las reglas (no las supera); menos transparente |

### Lectura para la defensa
- El **NER aporta valor real** en la extracción: generaliza a redacciones nuevas que la regex no anticipó (p. ej. "amerita complementar examen"), sin agregar reglas.
- La **Prueba A** es la más defendible: la IA hace lo que hace bien (encontrar el span en texto variable) y las reglas hacen lo que hacen bien (asignar categoría de forma transparente y auditable).
- La **Prueba B** demuestra un pipeline totalmente neuronal, pero es honesto reconocer que su clasificador se entrena con etiquetas derivadas de las reglas: no las supera, las imita.
- **Límite de datos**: el modelo se entrenó con corpus paraguayo; generalizará mejor que la regex, pero un corpus chileno sigue siendo la mejora definitiva.

### Guardar los modelos

In [17]:
trainer.save_model("../models/ner_recomendacion_final")
tokenizer.save_pretrained("../models/ner_recomendacion_final")
print("Modelo NER guardado en ../models/ner_recomendacion_final")
# (Prueba B) trainer_clf.save_model("../models/clasificador_recomendacion_final")

Modelo NER guardado en ../models/ner_recomendacion_final


## 13. Prueba de generalización real (informes chilenos externos)

**Este es el test honesto.** La métrica interna (F1 alto) mide el corpus paraguayo, homogéneo. Para saber si el modelo de verdad *aprendió a extraer* —y no memorizó— lo evaluamos con **informes chilenos reales**, de otra distribución, que no están en el corpus.

Si extrae bien aquí, es prueba incontestable de que generaliza. Si falla, es la evidencia honesta de la brecha corpus-despliegue (y el valor de esta prueba es, justamente, mostrarla en vez de esconderla detrás de un F1=1.0).

In [18]:
# Informes chilenos reales probados durante la validación (formatos externos).
# Cada uno con su recomendación esperada (o None si no trae recomendación).
informes_chilenos = [
    {
        "texto": ("impresion diagnostica. examen sin signos sospechosos de malignidad. "
                  "birads 2. acr c. dado el patron mamografico, se sugiere complementar "
                  "examen con ecografia."),
        "esperado": "complementar examen con ecografia",
    },
    {
        "texto": ("impresion. examen sin hallazgos sugerentes de malignidad. "
                  "recomiendo control mamografico y ecografico anual. bi-rads us 2"),
        "esperado": "recomiendo control mamografico y ecografico anual",
    },
    {
        "texto": ("hallazgos benignos. birads -us ii. examen sin hallazgos sospechosos."),
        "esperado": None,  # este informe NO trae recomendación explícita
    },
]

print("PRUEBA DE GENERALIZACIÓN — informes chilenos externos")
print("="*64)
for i, caso in enumerate(informes_chilenos, 1):
    extraido = extraer_recomendacion_ner(caso["texto"])
    print(f"\nInforme {i}:")
    print(f"  Esperado : {caso['esperado']}")
    print(f"  Extraído : {extraido!r}")
    if caso["esperado"] is None:
        print(f"  (Ideal: no extraer nada o marcar revisión)")

PRUEBA DE GENERALIZACIÓN — informes chilenos externos

Informe 1:
  Esperado : complementar examen con ecografia
  Extraído : 'dado el patron mamografico, se sugiere complementar examen con ecografia.'

Informe 2:
  Esperado : recomiendo control mamografico y ecografico anual
  Extraído : 'recomiendo control mamografico y ecografico anual.'

Informe 3:
  Esperado : None
  Extraído : ''
  (Ideal: no extraer nada o marcar revisión)


### Cómo interpretar y reportar estos resultados

- Si el modelo extrae bien las recomendaciones chilenas **con redacciones que la regex no tenía** (p. ej. "complementar examen con ecografia"), demuestra generalización real: aprendió el *concepto* de recomendación, no memorizó frases.
- Si falla en algún formato muy distinto, es la evidencia honesta de la brecha de datos.

**Para el informe/defensa**, reporta así:
> *"El modelo alcanza F1 cercano a 1.0 en el corpus, atribuible a su alta homogeneidad (la recomendación está al final en el 99% de los casos y existen ~250 recomendaciones distintas). Para evaluar generalización real, se probó con informes chilenos externos de formato no visto; [describir resultado]. Esto separa el desempeño intra-corpus del desempeño en el contexto de despliegue."*

Esta honestidad —distinguir un F1 inflado por homogeneidad de una generalización real— es exactamente el tipo de análisis crítico que se espera en un proyecto de IA.